# AI Summaries & Multimodal Queries

Search and face clustering give the app structure. What they cannot do is tell a story. A gallery of 200 beach photos from July 2023 is a pile of files; a sentence like "A warm week at the coast with Alice and the kids, mostly sunny, one rainy afternoon" is a memory. This notebook layers GPT-4o-mini on top of the existing stack to generate these narrative summaries and to let users query the library in natural language.

We cover two capabilities: (1) **batch photo description**: calling the vision API on a collection of photos to produce structured scene metadata, and (2) **narrative summarization**: assembling retrieved metadata into a RAG context and asking the model to write a 3-sentence summary. We then build a query planner that parses natural language into typed `SearchQuery` filters, enabling hybrid search that combines SQL date/person filtering with CLIP vector ranking.

## GPT-4o-mini Vision API

**Multimodal LLMs.** A vision-capable LLM extends the standard text token sequence with image tokens. A **vision encoder** (typically a CLIP-like ViT) divides the image into patches, projects each patch into the model's embedding dimension, and inserts the resulting patch token sequence into the context window. The autoregressive LLM then attends jointly over text and image tokens when generating its response. From the API perspective, this is invisible: we simply include an `image_url` content block alongside the text prompt.

<br>

**Model choice.** `gpt-4o-mini` is approximately $10\times$ cheaper per token than `gpt-4o` while retaining strong scene-description ability. For photo summarization — where precision matters less than fluency — the quality difference is negligible. A rough cost estimate: a 512×512 image costs approximately 170 input tokens at the low-detail setting; at $\approx \$0.00015$ per 1K tokens, describing 1,000 photos costs $\approx \$0.03$.

<br>

**API structure.** The vision call uses the standard `chat.completions.create` endpoint with a `content` list containing both an `image_url` block (pointing to a presigned S3 URL) and a `text` block:

```python
await client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": presigned_url, "detail": "low"}},
            {"type": "text",      "text": prompt},
        ],
    }],
    max_tokens=256,
)
```

The `detail` parameter controls resolution: `"low"` uses the fixed 512-token budget; `"high"` tiles the image and can cost up to 1,750 tokens. For a scene-description task, `"low"` is sufficient and an order of magnitude cheaper.

:::{.callout-note}
Presigned S3 URLs expire, typically in 15 minutes for the default `GetObject` presign. For batch inference where the job may run longer, either generate fresh URLs immediately before calling the API or upload photos to a temporary public prefix with a short lifecycle policy.

:::

Defining `describe_photo` with a mock `AsyncOpenAI` client:

In [ ]:
import asyncio
from unittest.mock import AsyncMock, MagicMock


def _make_mock_client(response_text: str):
    """Construct a minimal AsyncOpenAI mock returning a fixed completion."""
    choice = MagicMock()
    choice.message.content = response_text

    usage = MagicMock()
    usage.total_tokens = 210

    completion = MagicMock()
    completion.choices = [choice]
    completion.usage  = usage

    client = MagicMock()
    client.chat.completions.create = AsyncMock(return_value=completion)
    return client


SYSTEM_PROMPT = (
    "You are a photo archivist. Describe the photo in 2–3 sentences: "
    "what is happening, who is present (if identifiable), "
    "and the mood or setting."
)


async def describe_photo(
    presigned_url: str,
    client,
    prompt: str = "Describe this photo.",
) -> tuple[str, int]:
    """Return (description, total_tokens) for a single photo."""
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": presigned_url, "detail": "low"}},
                    {"type": "text",      "text": prompt},
                ],
            },
        ],
        max_tokens=256,
    )
    return resp.choices[0].message.content, resp.usage.total_tokens


mock_client = _make_mock_client(
    "A sunny afternoon at the beach. Two adults and a child are building a sandcastle "
    "near the water's edge. The mood is relaxed and playful."
)

desc, tokens = asyncio.run(
    describe_photo("https://s3.example.com/photo.jpg?presign=...", mock_client)
)
print(f"Description ({tokens} tokens):\n{desc}")

## Prompt Design for Photo Descriptions

**Zero-shot vs. few-shot.** Zero-shot prompting — just the system message and a `"Describe this photo."` user prompt — produces fluent natural-language descriptions but with inconsistent structure. For downstream use in RAG contexts we want machine-parseable output: a fixed set of fields we can reliably extract and index. **Few-shot** prompting with 2–3 examples closes the consistency gap, but a more robust solution is to ask the model for JSON output directly and validate it with a Pydantic model.

<br>

**Structured output.** Setting `response_format={"type": "json_object"}` in the API call instructs GPT-4o-mini to return valid JSON. Combined with a schema description in the prompt, this gives reliable structured extraction with no post-processing:

```python
prompt = (
    "Describe this photo as JSON with keys: "
    "scene (str), people_count (int), mood (str), location_hint (str or null)."
)
```

We then validate with `PhotoDescription.model_validate_json(resp.choices[0].message.content)`.

Defining the `PhotoDescription` Pydantic model and demonstrating structured extraction:

In [ ]:
import json
from pydantic import BaseModel


class PhotoDescription(BaseModel):
    scene:         str
    people_count:  int
    mood:          str
    location_hint: str | None = None


STRUCTURED_PROMPT = (
    "Describe this photo as JSON with the following keys: "
    "scene (string), people_count (integer), "
    "mood (string), location_hint (string or null)."
)


mock_json_response = json.dumps({
    "scene":         "beach with sandcastle construction",
    "people_count":  3,
    "mood":          "relaxed and playful",
    "location_hint": "ocean beach, likely summer",
})

structured_client = _make_mock_client(mock_json_response)


async def describe_photo_structured(
    presigned_url: str,
    client,
) -> PhotoDescription:
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": presigned_url, "detail": "low"}},
                    {"type": "text",      "text": STRUCTURED_PROMPT},
                ],
            },
        ],
        response_format={"type": "json_object"},
        max_tokens=256,
    )
    return PhotoDescription.model_validate_json(resp.choices[0].message.content)


photo_desc = asyncio.run(
    describe_photo_structured("https://s3.example.com/photo.jpg", structured_client)
)
print(photo_desc.model_dump_json(indent=2))

## Batch Inference for Cost Efficiency

**Naive sequential inference.** One API call per photo is the simplest approach, but it processes photos one at a time. At 500ms per call, 1,000 photos would take over 8 minutes. The bottleneck is network round-trip time, not GPU compute: the OpenAI API can handle many more concurrent requests than we typically issue.

<br>

**Concurrent batching.** We use `asyncio.gather` to fire multiple API calls simultaneously. An `asyncio.Semaphore(N)` caps the number of inflight requests at $N$, preventing us from exceeding the rate limit. GPT-4o-mini's default tier allows approximately 500 RPM (requests per minute) and 200K TPM (tokens per minute); with 10 concurrent requests at ~210 tokens each, we stay well within both limits.

<br>

**Rate limit handling.** A `429 Too Many Requests` response means we have exceeded the rate limit. The `tenacity` library makes retrying with exponential backoff straightforward:

```python
from tenacity import retry, stop_after_attempt, wait_exponential

@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=1, min=1, max=30))
async def describe_with_retry(url, client):
    return await describe_photo_structured(url, client)
```

**Cost tracking.** Every `ChatCompletion` response includes a `usage.total_tokens` field. We accumulate these across the batch and log the estimated cost at the end.

Implementing `batch_describe` with a semaphore and cost accumulation:

In [ ]:
import asyncio
import time
from typing import Any


COST_PER_1K_TOKENS = 0.00015  # USD, gpt-4o-mini input pricing


async def batch_describe(
    presigned_urls: list[str],
    client: Any,
    concurrency: int = 10,
) -> tuple[list[PhotoDescription], dict]:
    """Describe all photos concurrently; return descriptions and cost stats."""
    sem           = asyncio.Semaphore(concurrency)
    total_tokens  = 0
    lock          = asyncio.Lock()

    async def _describe_one(url: str) -> PhotoDescription:
        nonlocal total_tokens
        async with sem:
            desc = await describe_photo_structured(url, client)
            # In a real call we would read resp.usage.total_tokens;
            # here we simulate 210 tokens per photo.
            async with lock:
                total_tokens += 210
            return desc

    t0 = time.perf_counter()
    results = await asyncio.gather(*[_describe_one(url) for url in presigned_urls])
    elapsed = time.perf_counter() - t0

    stats = {
        "n_photos":     len(results),
        "total_tokens": total_tokens,
        "elapsed_s":    round(elapsed, 2),
        "est_cost_usd": round(total_tokens / 1000 * COST_PER_1K_TOKENS, 5),
    }
    return list(results), stats


# Simulate 20 photos
urls = [f"https://s3.example.com/photo_{i}.jpg" for i in range(20)]
descriptions, stats = asyncio.run(batch_describe(urls, structured_client))
print(f"Described {stats['n_photos']} photos in {stats['elapsed_s']}s")
print(f"Tokens: {stats['total_tokens']}  Estimated cost: ${stats['est_cost_usd']}")

## RAG over Photo Metadata

**Retrieval-Augmented Generation.** Asking GPT-4o-mini to summarize "summer 2023" without any grounding will produce plausible-sounding but fabricated details: specific dates, names, and places that the model has no access to. **RAG** prevents this: we first retrieve the actual photos and their metadata, format them as a context string, and then ask the model to summarize *only what is in the context*. The model's role is generation and synthesis, not recall.

<br>

**Retrieval step.** For a time-period summary, we filter `photos.taken_at` by date range to get candidate photos, then rank them by CLIP relevance to a short query (e.g., the period title) to select the top-$k$ most representative. We join with the `faces` table to resolve person names. The target context size is 50–100 photos, large enough to be representative and small enough to fit in the context window comfortably.

<br>

**Context format.** Each photo contributes one line:

```
[2023-07-15] Beach photo. People: Alice, Bob. Scene: sunny beach, children playing. Mood: joyful.
```

The generation prompt then wraps this context:

```
Write a 3-sentence narrative summary of this period for a personal photo album.
Base the summary only on the photos described below. Do not invent details.

{context}
```

:::{.callout-important}
Always include "Do not invent details" or equivalent grounding instruction. Without it, GPT-4o-mini occasionally interpolates plausible-sounding specifics (names, locations) not present in the context.

:::

Defining the `PhotoRecord` type and `build_context` function:

In [ ]:
from dataclasses import dataclass, field
from datetime import date


@dataclass
class PhotoRecord:
    photo_id:      str
    taken_at:      date
    scene:         str
    people:        list[str] = field(default_factory=list)
    mood:          str = ""
    location_hint: str | None = None


def build_context(photos: list[PhotoRecord]) -> str:
    """Format a list of PhotoRecords into a RAG context string."""
    lines = []
    for p in photos:
        people_str = ", ".join(p.people) if p.people else "unknown"
        loc_str    = f" Location: {p.location_hint}." if p.location_hint else ""
        lines.append(
            f"[{p.taken_at}] {p.scene}."
            f" People: {people_str}."
            f" Mood: {p.mood}.{loc_str}"
        )
    return "\n".join(lines)


# Example with 5 mock photos
sample_photos = [
    PhotoRecord("p1", date(2023, 7, 4),  "fireworks show",           ["Alice", "Bob"], "festive",   "city park"),
    PhotoRecord("p2", date(2023, 7, 8),  "beach day with sandcastle", ["Alice"],        "playful",   "ocean beach"),
    PhotoRecord("p3", date(2023, 7, 15), "barbecue in the backyard",  ["Bob", "Carol"], "relaxed",   None),
    PhotoRecord("p4", date(2023, 8, 2),  "hiking trail with dog",     ["Alice", "Bob"], "adventurous","mountain trail"),
    PhotoRecord("p5", date(2023, 8, 20), "rainy afternoon indoors",   [],              "cozy",      None),
]

context = build_context(sample_photos)
print(context)

## Narrative Summary API

**`Summary` model.** A summary pairs a user-supplied date range with a model-generated narrative and provenance metadata: how many photos were included (`photo_count`) and when the summary was last generated (`created_at`).

**Caching strategy.** LLM inference costs approximately 1,000 tokens and 2–4 seconds per summary. The correct strategy is to generate once and serve the cached narrative on subsequent requests. `created_at` lets the UI display "Last updated 3 days ago" and offer a "Regenerate" button. The regenerate path calls the same endpoint with `force=True`, which overwrites the cached row using an upsert on `(start_date, end_date)`.

**`photo_count` as a quality signal.** Exposing the photo count in the response lets the UI warn when a summary covers very few photos (< 5) and is therefore unreliable, or when the count has grown significantly since the last generation — a signal that the narrative is stale. A summary generated from 3 photos in a 90-day range should be flagged differently from one generated from 200 photos.

**One summary per date range.** The design uses an upsert on `(start_date, end_date)` — a second `POST` with the same range refreshes rather than duplicates. Storing multiple historical summaries per range is possible (change `id` to a sequence and remove the unique constraint) but rarely useful for a personal photo library.

Defining the Pydantic schemas for the summaries API:

In [ ]:
from datetime import date, datetime
from pydantic import BaseModel


class SummaryCreate(BaseModel):
    title:      str
    start_date: date
    end_date:   date


class SummaryRead(BaseModel):
    id:          int
    title:       str
    start_date:  date
    end_date:    date
    narrative:   str
    photo_count: int
    created_at:  datetime


print(SummaryCreate.model_fields)
print(SummaryRead.model_fields)

Full summaries router with a mock LLM call:

In [ ]:
import asyncio
from datetime import datetime
from fastapi import APIRouter
from fastapi.testclient import TestClient
from fastapi import FastAPI


GENERATION_PROMPT_TEMPLATE = (
    "Write a 3-sentence narrative summary of this period for a personal photo album. "
    "Base the summary only on the photos described below. Do not invent details.\n\n"
    "{context}"
)

# In-memory store for demo purposes
_summaries: dict[int, SummaryRead] = {}
_next_id = 1

summaries_router = APIRouter(prefix="/summaries", tags=["summaries"])


async def _generate_narrative(photos: list[PhotoRecord], client) -> str:
    context = build_context(photos)
    desc, _ = await describe_photo(
        presigned_url="",   # not used in text-only call
        client=client,
        prompt=GENERATION_PROMPT_TEMPLATE.format(context=context),
    )
    return desc


@summaries_router.post("/", response_model=SummaryRead)
async def create_summary(body: SummaryCreate) -> SummaryRead:
    global _next_id

    # Stub: return sample_photos filtered by date range
    photos = [
        p for p in sample_photos
        if body.start_date <= p.taken_at <= body.end_date
    ]
    narrative = await _generate_narrative(photos, mock_client)

    summary = SummaryRead(
        id=_next_id,
        title=body.title,
        start_date=body.start_date,
        end_date=body.end_date,
        narrative=narrative,
        photo_count=len(photos),
        created_at=datetime.utcnow(),
    )
    _summaries[_next_id] = summary
    _next_id += 1
    return summary


@summaries_router.get("/", response_model=list[SummaryRead])
async def list_summaries() -> list[SummaryRead]:
    return list(_summaries.values())


@summaries_router.get("/{summary_id}", response_model=SummaryRead)
async def get_summary(summary_id: int) -> SummaryRead:
    return _summaries[summary_id]


app = FastAPI()
app.include_router(summaries_router)
client = TestClient(app)

resp = client.post("/summaries/", json={
    "title":      "Summer 2023",
    "start_date": "2023-06-01",
    "end_date":   "2023-08-31",
})
print(resp.status_code)
print(resp.json())

## Query Planning

**Parsing natural language into structured filters.** A user who types "photos of Alice at the beach in summer 2022" is specifying multiple constraints simultaneously: a person name, a scene type, and a date range. Trying to handle this with keyword matching or regex is brittle — the same intent can be expressed dozens of ways. We instead use GPT-4o-mini as a **query parser**: we give it a `SearchQuery` schema and ask it to extract the structured filter from the natural language string.

<br>

**Hybrid search.** Once we have the structured `SearchQuery`, we apply two ranking passes:

1. **SQL filter**: restrict by `person_name` (join `faces`) and `date_range` (filter `taken_at`). This is exact and fast.
2. **CLIP vector rank**: encode the original query string, compute cosine similarity against the embedding column, and sort the SQL-filtered results by relevance.

Combining both passes gives substantially better recall than either alone: the SQL filter removes irrelevant photos efficiently; CLIP ranking surfaces the most visually relevant ones within the filtered set.

:::{.callout-caution}
The query parser can hallucinate person names or dates not present in the query. Always treat the parsed filter as a hint, not a guarantee. Validate date ranges before passing them to the SQL layer: an `end_date` before `start_date` will return an empty result set silently if not caught.

:::

Defining `SearchQuery` and `parse_query` with a mock OpenAI client:

In [ ]:
import asyncio
import json
from datetime import date
from pydantic import BaseModel


class SearchQuery(BaseModel):
    person_name:   str | None = None
    location_hint: str | None = None
    start_date:    date | None = None
    end_date:      date | None = None
    freeform:      str | None = None  # residual text for CLIP encoding


QUERY_PARSER_PROMPT = (
    "Extract a structured search filter from the user query. "
    "Return JSON with keys: person_name, location_hint, start_date (YYYY-MM-DD), "
    "end_date (YYYY-MM-DD), freeform. Use null for absent fields."
)


async def parse_query(natural_language: str, client) -> SearchQuery:
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": QUERY_PARSER_PROMPT},
            {"role": "user",   "content": natural_language},
        ],
        response_format={"type": "json_object"},
        max_tokens=128,
    )
    return SearchQuery.model_validate_json(resp.choices[0].message.content)


# Three representative query examples
queries_and_responses = [
    (
        "photos of Alice at the beach in summer 2022",
        {"person_name": "Alice", "location_hint": "beach",
         "start_date": "2022-06-01", "end_date": "2022-08-31", "freeform": "beach"},
    ),
    (
        "hiking pictures from last July",
        {"person_name": None, "location_hint": "hiking trail",
         "start_date": "2025-07-01", "end_date": "2025-07-31", "freeform": "hiking"},
    ),
    (
        "birthday party with Bob and Carol",
        {"person_name": "Bob", "location_hint": None,
         "start_date": None,  "end_date": None, "freeform": "birthday party Carol"},
    ),
]

for query_text, mock_response in queries_and_responses:
    parser_client = _make_mock_client(json.dumps(mock_response))
    sq = asyncio.run(parse_query(query_text, parser_client))
    print(f"Q: {query_text!r}")
    print(f"   → {sq.model_dump()}\n")

## Appendix: Caching and Deduplication

**Skipping already-described photos.** Re-running the indexing pipeline should not re-call the vision API for photos whose description has not changed. We track this with two columns on the `photos` table: `summary` (nullable text) and `indexed_at` (timestamp). When preparing the batch, we filter to photos where `summary IS NULL OR indexed_at < updated_at`. This makes the batch inference step idempotent.

<br>

**Semantic deduplication.** Near-duplicate photos — burst shots from the same second, or the same scene photographed twice — have very high CLIP cosine similarity ($> 0.98$). Including duplicates in the RAG context inflates the photo count without adding information and biases the summary toward whatever scene appears most frequently. We remove duplicates before building the context by greedily selecting photos with pairwise similarity below the threshold:

Greedy semantic deduplication over a batch of CLIP embeddings:

In [ ]:
import numpy as np


def deduplicate(embeddings: np.ndarray, threshold: float = 0.98) -> list[int]:
    """
    Greedy deduplication: keep an embedding only if it is not near-duplicate
    of any already-selected embedding.

    Parameters
    ----------
    embeddings : (N, D) float32 array of L2-normalised CLIP embeddings
    threshold  : cosine similarity above which two photos are duplicates

    Returns
    -------
    Indices of the selected (non-duplicate) photos.
    """
    selected: list[int] = []
    for i, emb in enumerate(embeddings):
        if not selected:
            selected.append(i)
            continue
        sims = embeddings[selected] @ emb                      # cosine sim (L2-normed)
        if sims.max() < threshold:
            selected.append(i)
    return selected


# Synthetic test: 10 embeddings, with items 1 and 2 being near-duplicates of item 0
rng = np.random.default_rng(42)
N, D = 10, 512
base = rng.standard_normal((N, D))
base[1] = base[0] + rng.standard_normal(D) * 0.01   # near-duplicate of 0
base[2] = base[0] + rng.standard_normal(D) * 0.01   # near-duplicate of 0
norms = np.linalg.norm(base, axis=1, keepdims=True)
embs  = (base / norms).astype(np.float32)

kept = deduplicate(embs, threshold=0.98)
print(f"Kept {len(kept)}/{N} photos after deduplication: indices {kept}")

---

■